In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("API key loaded:", bool(GEMINI_API_KEY))

API key loaded: True


In [ ]:
from google.colab import userdata

# Make sure GEMINI_API_KEY is wrapped in quotes
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Make sure your print statement string has quotes
print("API key loaded: ", bool(GEMINI_API_KEY))


API key loaded:  True


In [ ]:

!pip install -q datasets

from datasets import load_dataset
import json
import os

print("Loading HotpotQA...")

dataset = load_dataset("hotpotqa/hotpot_qa", "distractor")

print("Available splits:", dataset.keys())

validation = dataset["validation"]

print("Validation records:", len(validation))

output_path = "/content/hotpot_dev_distractor_v1.json"

print("Writing JSON file...")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(list(validation), f, ensure_ascii=False)

print("File created:", os.path.exists(output_path))
print("File size:", round(os.path.getsize(output_path) / (1024**2), 2), "MB")


Loading HotpotQA...
Available splits: dict_keys(['train', 'validation'])
Validation records: 7405
Writing JSON file...
File created: True
File size: 44.46 MB


In [ ]:

import json
import time
from google import genai
from google.colab import userdata

# ============================================================
# 1. AUTHENTICATION
# ============================================================

try:
    api_key = userdata.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("GEMINI_API_KEY missing from Colab Secrets.")

    client = genai.Client(api_key=api_key)

    print("API Configuration: SUCCESS")

except Exception as e:
    print("API Configuration FAILED:", e)
    raise


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = "/content/hotpot_dev_distractor_v1.json"

try:
    with open(file_path, "r", encoding="utf-8") as f:
        full_data = json.load(f)

    dataset = full_data[:2]  # Testing on 2 samples

    print("Dataset loaded successfully.")
    print("Total samples:", len(full_data))
    print("Test samples:", len(dataset))

except Exception as e:
    print("Dataset Loading FAILED:", e)
    raise


# ============================================================
# 3. MODEL CONFIGURATION
# ============================================================

models_to_test = [
    "gemini-3.8-flash",
    "gemini-3.5-flash-lite"
]

active_model = None


# ============================================================
# 4. TEST MODEL AVAILABILITY
# ============================================================

print("\n--- Testing Gemini Models ---")

for model in models_to_test:

    print(f"\nTesting model: {model}")

    try:
        response = client.models.generate_content(
            model=model,
            contents="Reply with exactly: CONNECTION_OK"
        )

        if response.text:
            print("SUCCESS:", response.text.strip())
            active_model = model
            break

    except Exception as e:
        print("FAILED:", str(e)[:300])

if active_model is None:
    raise RuntimeError(
        "All models failed. Check API availability and retry later."
    )

print("\nSelected model:", active_model)


# ============================================================
# 5. GENERATION FUNCTION WITH EXPONENTIAL BACKOFF
# ============================================================

def generate_with_retry(
    prompt,
    model_name,
    max_retries=5
):

    for attempt in range(max_retries):

        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if response.text:
                return response.text.strip()

            return "ERROR: Empty response"

        except Exception as e:

            error_str = str(e)

            if "503" in error_str or "429" in error_str:

                if attempt == max_retries - 1:
                    return f"ERROR: Retries exhausted: {error_str}"

                wait_time = min(5 * (2 ** attempt, 60))

                print(
                    f"Attempt {attempt+1}/{max_retries} failed. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                return f"ERROR: {error_str}"

    return "ERROR: Generation failed"


# ============================================================
# 6. BASELINE AGENT EXECUTION
# ============================================================

print("\n--- Running Baseline Agent ---")

results = []

for i, row in enumerate(dataset):

    question = row["question"]

    # Format HotpotQA context correctly
    context_str = ""

    for title, sentences in zip(
        row["context"]["title"],
        row["context"]["sentences"]
    ):

        context_str += (
            f"Title: {title}\n"
            f"Text: {' '.join(sentences)}\n\n"
        )

    # Baseline prompt
    prompt = f"""
You are a research agent.

Answer the question based ONLY on the provided context.

Context:
{context_str}

Question:
{question}

Provide step-by-step reasoning followed by a clearly
identified final answer.
"""

    print("\n" + "=" * 60)
    print(f"QUESTION {i+1}")
    print("=" * 60)

    print("Question:", question)

    answer = generate_with_retry(
        prompt=prompt,
        model_name=active_model
    )

    print("\nAgent Output:")
    print(answer)

    # Store experimental results
    results.append({
        "sample_id": i,
        "question": question,
        "ground_truth": row["answer"],
        "agent_output": answer,
        "model": active_model,
        "status": (
            "success"
            if not answer.startswith("ERROR:")
            else "failed"
        )
    })

    time.sleep(3)


# ============================================================
# 7. SAVE RESULTS
# ============================================================

output_file = "/content/baseline_results.json"

with open(output_file, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )

successful = sum(
    1 for r in results if r["status"] == "success"
)

failed = len(results) - successful

print("\n" + "=" * 60)
print("BASELINE EXPERIMENT COMPLETED")
print("=" * 60)

print("Selected model:", active_model)
print("Total questions:", len(results))
print("Successful:", successful)
print("Failed:", failed)
print("Results saved:", output_file)


API Configuration: SUCCESS
Dataset loaded successfully.
Total samples: 7405
Test samples: 2

--- Testing Gemini Models ---

Testing model: gemini-3.8-flash
FAILED: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Testing model: gemini-3.5-flash-lite
SUCCESS: CONNECTION_OK

Selected model: gemini-3.5-flash-lite

--- Running Baseline Agent ---

QUESTION 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?

Agent Output:
Step-by-step reasoning:
1. According to the text, Scott Derrickson is an "American director, screenwriter and producer" (indicating his nationality is American).
2. According to the text, Edward Davis Wood Jr. (Ed Wood) was an "American filmmaker, actor, writer, producer, and director" (indicating his nationality is American).
3. Both individuals share the same nationality (American).

Final Answer: Yes

QUEST

In [ ]:

import json
import time
import os
from google import genai
from google.colab import userdata

# ============================================================
# 1. AUTHENTICATE GEMINI API
# ============================================================

try:
    api_key = userdata.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("GEMINI_API_KEY not found in Colab Secrets.")

    client = genai.Client(api_key=api_key)

    print("Gemini API Authentication: SUCCESS")

except Exception as e:
    print("API Authentication FAILED:", e)
    raise


# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "gemini-3.5-flash-lite"

FILE_PATH = "/content/hotpot_dev_distractor_v1.json"

OUTPUT_FILE = "/content/baseline_results.json"

TEST_SAMPLES = 2

MAX_RETRIES = 5

BASE_DELAY = 5


# ============================================================
# 3. LOAD DATASET
# ============================================================

try:

    with open(FILE_PATH, "r", encoding="utf-8") as f:
        full_data = json.load(f)

    dataset = full_data[:TEST_SAMPLES]

    print("\nDataset loaded successfully.")
    print("Total samples:", len(full_data))
    print("Testing samples:", len(dataset))

except Exception as e:

    print("Dataset Loading FAILED:", e)
    raise


# ============================================================
# 4. GENERATION FUNCTION WITH EXPONENTIAL BACKOFF
# ============================================================

def generate_with_retry(
    prompt,
    model_name=MODEL_NAME,
    max_retries=MAX_RETRIES,
    base_delay=BASE_DELAY
):

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if response.text:
                return {
                    "status": "success",
                    "output": response.text.strip(),
                    "error": None
                }

            return {
                "status": "failed",
                "output": None,
                "error": "Empty response from Gemini"
            }

        except Exception as e:

            error_str = str(e)

            retryable = (
                "503" in error_str
                or "429" in error_str
                or "500" in error_str
            )

            if retryable and attempt < max_retries - 1:

                wait_time = min(
                    base_delay * (2 ** attempt),
                    60
                )

                print(
                    f"Attempt {attempt + 1}/{max_retries} failed. "
                    f"Retrying in {wait_time}s..."
                )

                time.sleep(wait_time)

            else:

                return {
                    "status": "failed",
                    "output": None,
                    "error": error_str
                }

    return {
        "status": "failed",
        "output": None,
        "error": "Maximum retries exceeded"
    }


# ============================================================
# 5. BASELINE AGENT EXECUTION
# ============================================================

print("\n--- Running Baseline Agent Prototype ---")

results = []

for i, row in enumerate(dataset):

    question = row["question"]

    ground_truth = row["answer"]

    # --------------------------------------------------------
    # FORMAT HOTPOTQA CONTEXT
    # --------------------------------------------------------

    context_str = ""

    titles = row["context"]["title"]

    paragraphs = row["context"]["sentences"]

    for title, sentences in zip(titles, paragraphs):

        context_str += (
            f"Title: {title}\n"
            f"Text: {' '.join(sentences)}\n\n"
        )

    # --------------------------------------------------------
    # BASELINE PROMPT
    # --------------------------------------------------------

    prompt = f"""
You are a research agent.

Answer the question using ONLY the provided context.

Do not use external knowledge.

Context:
{context_str}

Question:
{question}

Provide a concise explanation of the evidence used.
Then provide the final answer in this format:

FINAL ANSWER: <answer>
"""

    # --------------------------------------------------------
    # GENERATE ANSWER
    # --------------------------------------------------------

    print("\n" + "=" * 60)

    print(f"QUESTION {i + 1}")

    print("=" * 60)

    print("Question:", question)

    generation = generate_with_retry(prompt)

    answer = generation["output"]

    print("\nAgent Output:")

    print(answer if answer else generation["error"])

    # --------------------------------------------------------
    # SAVE INDIVIDUAL RESULT
    # --------------------------------------------------------

    result = {

        "sample_id": i,

        "question": question,

        "ground_truth": ground_truth,

        "agent_output": answer,

        "model": MODEL_NAME,

        "status": generation["status"],

        "error": generation["error"]

    }

    results.append(result)

    # Delay between requests
    time.sleep(2)


# ============================================================
# 6. SAVE EXPERIMENT RESULTS
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 7. EXPERIMENT SUMMARY
# ============================================================

successful = sum(
    1 for result in results
    if result["status"] == "success"
)

failed = len(results) - successful

print("\n" + "=" * 60)

print("BASELINE EXPERIMENT COMPLETED")

print("=" * 60)

print("Model:", MODEL_NAME)

print("Total questions:", len(results))

print("Successful:", successful)

print("Failed:", failed)

print("Results saved to:", OUTPUT_FILE)


Gemini API Authentication: SUCCESS

Dataset loaded successfully.
Total samples: 7405
Testing samples: 2

--- Running Baseline Agent Prototype ---

QUESTION 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?

Agent Output:
Based on the provided text, Scott Derrickson is described as an "American director, screenwriter and producer," and Edward Davis Wood Jr. (Ed Wood) is described as an "American filmmaker, actor, writer, producer, and director." Therefore, both individuals share the same nationality (American).

FINAL ANSWER: Yes

QUESTION 2
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?

Agent Output:
Based on the provided context, Shirley Temple portrayed Corliss Archer in the 1945 film "Kiss and Tell" (as well as the 1949 film "A Kiss for Corliss"). According to her biography, as an adult she held the government positions of United States ambassador to Ghana and to Czechoslovakia, and she also serv

In [ ]:

import json
import time
import pandas as pd
from google import genai
from google.colab import userdata

# ============================================================
# 1. AUTHENTICATE GEMINI API
# ============================================================

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY missing from Colab Secrets.")

client = genai.Client(api_key=api_key)

print("Gemini API Authentication: SUCCESS")


# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "gemini-3.5-flash-lite"

FILE_PATH = "/content/hotpot_dev_distractor_v1.json"

OUTPUT_CSV = "/content/baseline_results.csv"

OUTPUT_JSON = "/content/baseline_results.json"

TEST_SAMPLES = 10

MAX_RETRIES = 5

BASE_DELAY = 5


# ============================================================
# 3. LOAD DATASET
# ============================================================

with open(FILE_PATH, "r", encoding="utf-8") as f:
    full_data = json.load(f)

dataset = full_data[:TEST_SAMPLES]

print("\nDataset loaded successfully.")
print("Total samples:", len(full_data))
print("Testing samples:", len(dataset))


# ============================================================
# 4. GENERATION FUNCTION WITH EXPONENTIAL BACKOFF
# ============================================================

def generate_with_retry(
    prompt,
    model_name=MODEL_NAME,
    max_retries=MAX_RETRIES,
    base_delay=BASE_DELAY
):

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if response.text:

                return {
                    "status": "success",
                    "output": response.text.strip(),
                    "error": None
                }

            return {
                "status": "failed",
                "output": None,
                "error": "Empty response from Gemini"
            }

        except Exception as e:

            error_str = str(e)

            retryable = (
                "503" in error_str
                or "429" in error_str
                or "500" in error_str
            )

            if retryable and attempt < max_retries - 1:

                wait_time = min(
                    base_delay * (2 ** attempt),
                    60
                )

                print(
                    f"Attempt {attempt + 1}/{max_retries} failed. "
                    f"Retrying in {wait_time}s..."
                )

                time.sleep(wait_time)

            else:

                return {
                    "status": "failed",
                    "output": None,
                    "error": error_str
                }

    return {
        "status": "failed",
        "output": None,
        "error": "Maximum retries exceeded"
    }


# ============================================================
# 5. BASELINE EVALUATION
# ============================================================

print("\n--- Running Formal Baseline Evaluation ---")

results = []

correct_count = 0

successful_count = 0

failed_count = 0

for i, row in enumerate(dataset):

    question = row["question"]

    ground_truth = row["answer"]

    # --------------------------------------------------------
    # FORMAT HOTPOTQA CONTEXT
    # --------------------------------------------------------

    context_str = ""

    titles = row["context"]["title"]

    paragraphs = row["context"]["sentences"]

    for title, sentences in zip(titles, paragraphs):

        context_str += (
            f"Title: {title}\n"
            f"Text: {' '.join(sentences)}\n\n"
        )

    # --------------------------------------------------------
    # BASELINE PROMPT
    # --------------------------------------------------------

    prompt = f"""
You are a research agent.

Answer the question using ONLY the provided context.
Do not use external knowledge.

Context:
{context_str}

Question:
{question}

Provide concise evidence-based reasoning.

Finally, on a new line, provide ONLY the final short answer
prefixed with "FINAL ANSWER:".

Example:
FINAL ANSWER: American
"""

    # --------------------------------------------------------
    # GENERATE ANSWER
    # --------------------------------------------------------

    print("\n" + "=" * 60)

    print(f"Processing Q{i + 1}/{len(dataset)}")

    print("=" * 60)

    print("Question:", question)

    generation = generate_with_retry(prompt)

    agent_output = generation["output"]

    status = generation["status"]

    prediction = None

    is_correct = None

    # --------------------------------------------------------
    # EXTRACT FINAL ANSWER
    # --------------------------------------------------------

    if status == "success":

        successful_count += 1

        if "FINAL ANSWER:" in agent_output:

            prediction = (
                agent_output
                .split("FINAL ANSWER:")[-1]
                .strip()
                .lower()
            )

        else:

            prediction = agent_output.strip().lower()

        # Preliminary substring evaluation
        is_correct = (
            ground_truth.lower() in prediction
        )

        if is_correct:

            correct_count += 1

        print("\nGround Truth:", ground_truth)

        print("Prediction:", prediction)

        print("Correct:", is_correct)

    else:

        failed_count += 1

        print("\nGeneration FAILED:", generation["error"])

    # --------------------------------------------------------
    # SAVE INDIVIDUAL RESULT
    # --------------------------------------------------------

    results.append({

        "id": row.get("_id", row.get("id", i)),

        "question": question,

        "ground_truth": ground_truth,

        "agent_output": agent_output,

        "extracted_prediction": prediction,

        "is_correct": is_correct,

        "status": status,

        "error": generation["error"],

        "model": MODEL_NAME

    })

    # Rate-limit pacing
    time.sleep(2)


# ============================================================
# 6. EXPORT RESULTS
# ============================================================

df = pd.DataFrame(results)

df.to_csv(OUTPUT_CSV, index=False)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 7. FINAL REPORT
# ============================================================

accuracy = (
    correct_count / successful_count * 100
    if successful_count > 0
    else 0
)

print("\n" + "=" * 60)

print("FORMAL BASELINE EVALUATION COMPLETED")

print("=" * 60)

print("Model:", MODEL_NAME)

print("Total samples:", len(dataset))

print("Successful generations:", successful_count)

print("Failed generations:", failed_count)

print("Correct answers:", correct_count)

print(
    f"Preliminary Accuracy: "
    f"{accuracy:.2f}% "
    f"({correct_count}/{successful_count})"
)

print("\nCSV saved:", OUTPUT_CSV)

print("JSON saved:", OUTPUT_JSON)


Gemini API Authentication: SUCCESS

Dataset loaded successfully.
Total samples: 7405
Testing samples: 10

--- Running Formal Baseline Evaluation ---

Processing Q1/10
Question: Were Scott Derrickson and Ed Wood of the same nationality?

Ground Truth: yes
Prediction: yes
Correct: True

Processing Q2/10
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?

Ground Truth: Chief of Protocol
Prediction: united states ambassador to ghana and to czechoslovakia and chief of protocol of the united states
Correct: True

Processing Q3/10
Question: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?

Ground Truth: Animorphs
Prediction: animorphs
Correct: True

Processing Q4/10
Question: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?

Ground Truth: no
Prediction: no
Correct: True

Processing Q5/10
Question: Th

In [21]:

import json
import time
import re
import os
import pandas as pd
from google import genai
from google.colab import userdata

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "gemini-3.5-flash-lite"

DATASET_PATH = "/content/hotpot_dev_distractor_v1.json"

BASELINE_PATH = "/content/baseline_results.csv"

EXPERIMENT_PATH = "/content/experimental_results.csv"

FAILURE_PATH = "/content/over_corrections_analysis.csv"

COMPARISON_PATH = "/content/comparison_results.csv"

TEST_SAMPLES = 10

MAX_RETRIES = 5

BASE_DELAY = 5

# ============================================================
# 2. GEMINI AUTHENTICATION
# ============================================================

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY missing.")

client = genai.Client(api_key=api_key)

print("Gemini Authentication: SUCCESS")


# ============================================================
# 3. GENERATION FUNCTION
# ============================================================

def generate_with_retry(prompt):

    for attempt in range(MAX_RETRIES):

        try:

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            if response.text:

                return {
                    "status": "success",
                    "output": response.text.strip(),
                    "error": None
                }

            return {
                "status": "failed",
                "output": None,
                "error": "Empty response"
            }

        except Exception as e:

            error = str(e)

            retryable = any(
                code in error
                for code in ["429", "500", "503"]
            )

            if retryable and attempt < MAX_RETRIES - 1:

                wait = min(BASE_DELAY * (2 ** attempt), 60)

                print(f"Retrying after {wait}s")

                time.sleep(wait)

            else:

                return {
                    "status": "failed",
                    "output": None,
                    "error": error
                }

    return {
        "status": "failed",
        "output": None,
        "error": "Max retries exceeded"
    }


# ============================================================
# 4. ANSWER EXTRACTION
# ============================================================

def extract_answer(text):

    if not text:
        return ""

    match = re.search(
        r"FINAL ANSWER:\s*(.*)",
        text,
        re.IGNORECASE
    )

    if match:

        return match.group(1).strip().lower()

    return text.strip().lower()


# ============================================================
# 5. LOAD DATASET
# ============================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:

    dataset = json.load(f)[:TEST_SAMPLES]

print("Experimental samples:", len(dataset))


# ============================================================
# 6. EXPERIMENTAL AGENT
# ============================================================

experimental_results = []

for i, row in enumerate(dataset):

    question = row["question"]

    ground_truth = row["answer"]

    context = ""

    for title, sentences in zip(
        row["context"]["title"],
        row["context"]["sentences"]
    ):

        context += (
            f"Title: {title}\n"
            f"Text: {' '.join(sentences)}\n\n"
        )

    print("\n" + "=" * 70)

    print(f"Experimental Question {i+1}/{len(dataset)}")

    print(question)

    # --------------------------------------------------------
    # PHASE 1: INITIAL DRAFT
    # --------------------------------------------------------

    draft_prompt = f"""
You are a research QA agent.

Answer the question using ONLY the supplied context.

Context:
{context}

Question:
{question}

Give concise reasoning and finish with:
FINAL ANSWER: <short answer>
"""

    draft_generation = generate_with_retry(draft_prompt)

    if draft_generation["status"] != "success":

        experimental_results.append({

            "id": row.get("_id", row.get("id", i)),

            "question": question,

            "ground_truth": ground_truth,

            "draft_output": None,

            "monitor_output": None,

            "agent_output": None,

            "extracted_prediction": None,

            "is_correct": None,

            "status": "draft_failed",

            "error": draft_generation["error"],

            "model": MODEL_NAME

        })

        continue

    draft_output = draft_generation["output"]

    draft_answer = extract_answer(draft_output)

    print("\nPhase 1 Draft:", draft_answer)

    # --------------------------------------------------------
    # PHASE 2: MONITOR / REVISION
    # --------------------------------------------------------

    monitor_prompt = f"""
You are an independent answer verification and revision agent.

Your task is to inspect the initial answer and correct it ONLY
if the supplied evidence demonstrates an actual error.

Do not change an answer merely because another answer sounds
more plausible.

Do not invent evidence.
Do not use external knowledge.
Preserve a correct answer.

CONTEXT:
{context}

QUESTION:
{question}

INITIAL DRAFT:
{draft_output}

Instructions:
1. Check whether the draft answers the actual question.
2. Verify its answer against the context.
3. If correct, preserve the answer.
4. If incorrect, replace it with the context-supported answer.
5. Avoid unnecessary changes.

Finish with:
FINAL ANSWER: <short answer>
"""

    monitor_generation = generate_with_retry(monitor_prompt)

    if monitor_generation["status"] != "success":

        experimental_results.append({

            "id": row.get("_id", row.get("id", i)),

            "question": question,

            "ground_truth": ground_truth,

            "draft_output": draft_output,

            "monitor_output": None,

            "agent_output": None,

            "extracted_prediction": None,

            "is_correct": None,

            "status": "monitor_failed",

            "error": monitor_generation["error"],

            "model": MODEL_NAME

        })

        continue

    monitor_output = monitor_generation["output"]

    final_answer = extract_answer(monitor_output)

    # --------------------------------------------------------
    # EVALUATION
    # --------------------------------------------------------

    is_correct = ground_truth.lower() in final_answer

    print("Ground Truth:", ground_truth)

    print("Monitor Final:", final_answer)

    print("Correct:", is_correct)

    # --------------------------------------------------------
    # SAVE RESULT
    # --------------------------------------------------------

    experimental_results.append({

        "id": row.get("_id", row.get("id", i)),

        "question": question,

        "ground_truth": ground_truth,

        "draft_output": draft_output,

        "monitor_output": monitor_output,

        "agent_output": monitor_output,

        "extracted_prediction": final_answer,

        "is_correct": is_correct,

        "status": "success",

        "error": None,

        "model": MODEL_NAME

    })

    time.sleep(2)


# ============================================================
# 7. SAVE EXPERIMENTAL RESULTS
# ============================================================

df_exp = pd.DataFrame(experimental_results)

df_exp.to_csv(EXPERIMENT_PATH, index=False)

print("\nExperimental results saved:", EXPERIMENT_PATH)

print("Experimental records:", len(df_exp))


# ============================================================
# 8. FAILURE ANALYSIS
# ============================================================

df_base = pd.read_csv(BASELINE_PATH)

# Only compare successfully evaluated samples
df_exp_valid = df_exp[
    (df_exp["status"] == "success")
    &
    (df_exp["is_correct"].notna())
].copy()

df_base["is_correct"] = (
    df_base["is_correct"].astype(str).str.lower() == "true"
)

df_exp_valid["is_correct"] = (
    df_exp_valid["is_correct"].astype(str).str.lower() == "true"
)

df_merged = pd.merge(
    df_base,
    df_exp_valid,
    on="question",
    suffixes=("_base", "_exp"),
    how="inner",
    validate="one_to_one"
)

# ------------------------------------------------------------
# OVER-CORRECTIONS
# ------------------------------------------------------------

degradations = df_merged[
    (df_merged["is_correct_base"] == True)
    &
    (df_merged["is_correct_exp"] == False)
]

# ------------------------------------------------------------
# RECOVERIES
# ------------------------------------------------------------

recoveries = df_merged[
    (df_merged["is_correct_base"] == False)
    &
    (df_merged["is_correct_exp"] == True)
]

# ------------------------------------------------------------
# UNCHANGED CASES
# ------------------------------------------------------------

unchanged_correct = df_merged[
    (df_merged["is_correct_base"] == True)
    &
    (df_merged["is_correct_exp"] == True)
]

unchanged_incorrect = df_merged[
    (df_merged["is_correct_base"] == False)
    &
    (df_merged["is_correct_exp"] == False)
]


# ============================================================
# 9. DISPLAY OVER-CORRECTIONS
# ============================================================

print("\n" + "=" * 70)

print("OVER-CORRECTION ANALYSIS")

print("=" * 70)

print("Over-corrections:", len(degradations))

for _, r in degradations.iterrows():

    print("\nQuestion:", r["question"])

    print("Ground Truth:", r["ground_truth_base"])

    print("Baseline Correct:", r["extracted_prediction_base"])

    print("Phase 1 Draft:", r["draft_output"])

    print("Phase 2 Final:", r["extracted_prediction_exp"])

    print("-" * 70)


# ============================================================
# 10. METRICS
# ============================================================

total = len(df_merged)

baseline_correct = df_merged["is_correct_base"].sum()

experimental_correct = df_merged["is_correct_exp"].sum()

baseline_accuracy = (
    baseline_correct / total * 100 if total else 0
)

experimental_accuracy = (
    experimental_correct / total * 100 if total else 0
)

accuracy_change = experimental_accuracy - baseline_accuracy

over_correction_rate = (
    len(degradations) / baseline_correct * 100
    if baseline_correct else 0
)

baseline_wrong = total - baseline_correct

recovery_rate = (
    len(recoveries) / baseline_wrong * 100
    if baseline_wrong else 0
)


# ============================================================
# 11. SAVE ANALYSIS
# ============================================================

degradations.to_csv(FAILURE_PATH, index=False)

comparison = pd.DataFrame({

    "Metric": [
        "Matched Samples",
        "Baseline Accuracy",
        "Experimental Accuracy",
        "Accuracy Change",
        "Over-Corrections",
        "Recoveries",
        "Unchanged Correct",
        "Unchanged Incorrect",
        "Over-Correction Rate",
        "Recovery Rate"
    ],

    "Value": [
        total,
        round(baseline_accuracy, 2),
        round(experimental_accuracy, 2),
        round(accuracy_change, 2),
        len(degradations),
        len(recoveries),
        len(unchanged_correct),
        len(unchanged_incorrect),
        round(over_correction_rate, 2),
        round(recovery_rate, 2)
    ]
})

comparison.to_csv(COMPARISON_PATH, index=False)


# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)

print("BASELINE VS EXPERIMENTAL RESULTS")

print("=" * 70)

print("Matched samples:", total)

print(f"Baseline Accuracy: {baseline_accuracy:.2f}%")

print(f"Experimental Accuracy: {experimental_accuracy:.2f}%")

print(f"Accuracy Change: {accuracy_change:+.2f} percentage points")

print("Over-corrections:", len(degradations))

print("Recoveries:", len(recoveries))

print("Unchanged correct:", len(unchanged_correct))

print("Unchanged incorrect:", len(unchanged_incorrect))

print(f"Over-correction rate: {over_correction_rate:.2f}%")

print(f"Recovery rate: {recovery_rate:.2f}%")

print("\nSaved files:")

print(EXPERIMENT_PATH)

print(FAILURE_PATH)

print(COMPARISON_PATH)

print("\nEXPERIMENT COMPLETED")


Gemini Authentication: SUCCESS
Experimental samples: 10

Experimental Question 1/10
Were Scott Derrickson and Ed Wood of the same nationality?

Phase 1 Draft: yes
Ground Truth: yes
Monitor Final: yes
Correct: True

Experimental Question 2/10
What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?

Phase 1 Draft: united states ambassador to ghana and to czechoslovakia, and chief of protocol of the united states
Ground Truth: Chief of Protocol
Monitor Final: united states ambassador to ghana and to czechoslovakia, and chief of protocol of the united states
Correct: True

Experimental Question 3/10
What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?

Phase 1 Draft: animorphs
Ground Truth: Animorphs
Monitor Final: animorphs
Correct: True

Experimental Question 4/10
Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?



In [22]:

import json
import time
import re
import string
import pandas as pd
from collections import Counter
from google import genai
from google.colab import userdata

# ============================================================
# 1. AUTHENTICATION
# ============================================================

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY missing from Colab Secrets.")

client = genai.Client(api_key=api_key)

print("Gemini API Authentication: SUCCESS")


# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "gemini-3.5-flash-lite"

FILE_PATH = "/content/hotpot_dev_distractor_v1.json"

OUTPUT_CSV = "/content/experimental_results.csv"
OUTPUT_JSON = "/content/experimental_results.json"

TEST_SAMPLES = 10
MAX_RETRIES = 5
BASE_DELAY = 15


# ============================================================
# 3. LOAD DATASET
# ============================================================

with open(FILE_PATH, "r", encoding="utf-8") as f:
    full_data = json.load(f)

dataset = full_data[:TEST_SAMPLES]

print("Dataset loaded:", len(dataset))


# ============================================================
# 4. GENERATION FUNCTION WITH RETRY
# ============================================================

def generate_with_retry(prompt):

    for attempt in range(MAX_RETRIES):

        try:

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            if response.text:
                return {
                    "status": "success",
                    "output": response.text.strip(),
                    "error": None
                }

            return {
                "status": "failed",
                "output": None,
                "error": "Empty Gemini response"
            }

        except Exception as e:

            error_str = str(e)

            retryable = any(
                code in error_str
                for code in ["503", "429", "500"]
            )

            if retryable and attempt < MAX_RETRIES - 1:

                wait = min(BASE_DELAY * (2 ** attempt), 120)

                print(
                    f"Retry {attempt + 1}/{MAX_RETRIES}: "
                    f"Waiting {wait}s"
                )

                time.sleep(wait)

            else:

                return {
                    "status": "failed",
                    "output": None,
                    "error": error_str
                }

    return {
        "status": "failed",
        "output": None,
        "error": "Maximum retries exceeded"
    }


# ============================================================
# 5. ANSWER EXTRACTION
# ============================================================

def extract_answer(text, marker):

    if not text:
        return ""

    match = re.search(
        rf"{marker}\s*(.*)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return text.strip()


# ============================================================
# 6. HOTPOTQA NORMALIZATION
# ============================================================

def normalize_answer(answer):

    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def remove_punctuation(text):
        return "".join(
            ch for ch in text
            if ch not in string.punctuation
        )

    def white_space_fix(text):
        return " ".join(text.split())

    answer = answer.lower()

    answer = remove_punctuation(answer)

    answer = remove_articles(answer)

    answer = white_space_fix(answer)

    return answer


def exact_match(prediction, ground_truth):

    return int(
        normalize_answer(prediction)
        == normalize_answer(ground_truth)
    )


def token_f1(prediction, ground_truth):

    pred_tokens = normalize_answer(prediction).split()

    gold_tokens = normalize_answer(ground_truth).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)

    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:

        return float(pred_tokens == gold_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)

    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


# ============================================================
# 7. EXPERIMENTAL INTERVENTION
# ============================================================

print("\n--- Experimental Intervention ---")

experimental_results = []

correct_count_exp = 0
successful_count = 0
failed_count = 0

em_scores = []
f1_scores = []

for i, row in enumerate(dataset):

    question = row["question"]

    ground_truth = row["answer"]

    # --------------------------------------------------------
    # CORRECT CONTEXT FORMATTING
    # --------------------------------------------------------

    context_str = ""

    titles = row["context"]["title"]

    paragraphs = row["context"]["sentences"]

    for title, sentences in zip(titles, paragraphs):

        context_str += (
            f"Title: {title}\n"
            f"Text: {' '.join(sentences)}\n\n"
        )

    print("\n" + "=" * 70)

    print(f"Processing Q{i+1}/{len(dataset)}")

    print("Question:", question)

    # --------------------------------------------------------
    # PHASE 1: DRAFT GENERATION
    # --------------------------------------------------------

    draft_prompt = f"""
You are a research QA agent.

Answer the question using ONLY the provided context.

Context:
{context_str}

Question:
{question}

Provide concise reasoning.

On a new line, provide the short answer:
DRAFT ANSWER: <answer>
"""

    print("Step 1: Drafting...")

    draft_generation = generate_with_retry(draft_prompt)

    if draft_generation["status"] != "success":

        failed_count += 1

        experimental_results.append({

            "id": row.get("_id", row.get("id", i)),
            "question": question,
            "ground_truth": ground_truth,
            "draft_output": None,
            "final_output": None,
            "extracted_prediction": None,
            "exact_match": None,
            "f1": None,
            "is_correct": None,
            "status": "draft_failed",
            "error": draft_generation["error"],
            "model": MODEL_NAME

        })

        continue

    draft_output = draft_generation["output"]

    draft_answer = extract_answer(
        draft_output,
        "DRAFT ANSWER:"
    )

    print("Draft:", draft_answer)

    time.sleep(5)

    # --------------------------------------------------------
    # PHASE 2: MONITORING / VERIFICATION
    # --------------------------------------------------------

    verify_prompt = f"""
You are an independent Metacognitive Monitor.

Verify the draft answer against the provided context.

Context:
{context_str}

Question:
{question}

Agent's Draft:
{draft_output}

Instructions:
1. Check whether the draft answers the question.
2. Verify the answer using the context.
3. Correct it if evidence demonstrates an error.
4. Preserve it if it is already correct.
5. Do not invent facts.
6. Do not use external knowledge.
7. Return a concise final answer.

On a new line, provide:
FINAL ANSWER: <answer>
"""

    print("Step 2: Monitoring...")

    monitor_generation = generate_with_retry(verify_prompt)

    if monitor_generation["status"] != "success":

        failed_count += 1

        experimental_results.append({

            "id": row.get("_id", row.get("id", i)),
            "question": question,
            "ground_truth": ground_truth,
            "draft_output": draft_output,
            "final_output": None,
            "extracted_prediction": None,
            "exact_match": None,
            "f1": None,
            "is_correct": None,
            "status": "monitor_failed",
            "error": monitor_generation["error"],
            "model": MODEL_NAME

        })

        continue

    final_output = monitor_generation["output"]

    prediction = extract_answer(
        final_output,
        "FINAL ANSWER:"
    )

    # --------------------------------------------------------
    # PHASE 3: EVALUATION
    # --------------------------------------------------------

    em = exact_match(prediction, ground_truth)

    f1 = token_f1(prediction, ground_truth)

    is_correct = bool(em)

    successful_count += 1

    correct_count_exp += em

    em_scores.append(em)

    f1_scores.append(f1)

    print("Ground Truth:", ground_truth)

    print("Final Answer:", prediction)

    print("Exact Match:", em)

    print(f"Token F1: {f1:.4f}")

    # --------------------------------------------------------
    # PHASE 4: STORE RESULTS
    # --------------------------------------------------------

    experimental_results.append({

        "id": row.get("_id", row.get("id", i)),

        "question": question,

        "ground_truth": ground_truth,

        "draft_output": draft_output,

        "draft_prediction": draft_answer,

        "final_output": final_output,

        "extracted_prediction": prediction,

        "exact_match": em,

        "f1": f1,

        "is_correct": is_correct,

        "status": "success",

        "error": None,

        "model": MODEL_NAME

    })

    time.sleep(10)


# ============================================================
# 8. EXPORT RESULTS
# ============================================================

df_exp = pd.DataFrame(experimental_results)

df_exp.to_csv(OUTPUT_CSV, index=False)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:

    json.dump(
        experimental_results,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 9. FINAL REPORT
# ============================================================

accuracy_exp = (
    correct_count_exp / successful_count * 100
    if successful_count > 0
    else 0
)

average_f1 = (
    sum(f1_scores) / len(f1_scores)
    if f1_scores
    else 0
)

print("\n" + "=" * 70)

print("EXPERIMENTAL EVALUATION COMPLETED")

print("=" * 70)

print("Model:", MODEL_NAME)

print("Total samples:", len(dataset))

print("Successful samples:", successful_count)

print("Failed samples:", failed_count)

print("Correct answers:", correct_count_exp)

print(f"Exact Match Accuracy: {accuracy_exp:.2f}%")

print(f"Average Token F1: {average_f1:.4f}")

print("\nSaved files:")

print(OUTPUT_CSV)

print(OUTPUT_JSON)


Gemini API Authentication: SUCCESS
Dataset loaded: 10

--- Experimental Intervention ---

Processing Q1/10
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Step 1: Drafting...
Draft: Yes
Step 2: Monitoring...
Ground Truth: yes
Final Answer: Yes
Exact Match: 1
Token F1: 1.0000

Processing Q2/10
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Step 1: Drafting...
Draft: United States ambassador to Ghana, United States ambassador to Czechoslovakia, and Chief of Protocol of the United States
Step 2: Monitoring...
Ground Truth: Chief of Protocol
Final Answer: United States ambassador to Ghana, United States ambassador to Czechoslovakia, and Chief of Protocol of the United States
Exact Match: 0
Token F1: 0.3000

Processing Q3/10
Question: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Step 1: Drafting...
D

In [23]:

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD RESULTS
# ============================================================
base = pd.read_csv("baseline_results.csv")
exp = pd.read_csv("experimental_results.csv")

# ============================================================
# 2. STANDARDIZE ID COLUMN
# ============================================================
def standardize_id(df, name):
    df = df.copy()

    if "_id" not in df.columns:
        if "id" in df.columns:
            df["_id"] = df["id"]
        else:
            raise KeyError(f"{name}: Neither '_id' nor 'id' exists.")

    if df["_id"].isna().any():
        raise ValueError(f"{name}: Missing IDs detected.")

    df["_id"] = df["_id"].astype(str).str.strip()

    if df["_id"].duplicated().any():
        raise ValueError(f"{name}: Duplicate IDs detected.")

    return df

base = standardize_id(base, "Baseline")
exp = standardize_id(exp, "Experimental")

# ============================================================
# 3. SAFE BOOLEAN CONVERSION
# ============================================================
def parse_correctness(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if isinstance(value, (int, float)):
        if value == 1:
            return True
        if value == 0:
            return False

    value = str(value).strip().lower()

    if value in ["true", "1", "yes"]:
        return True
    if value in ["false", "0", "no"]:
        return False

    return np.nan

base["is_correct"] = base["is_correct"].apply(parse_correctness)
exp["is_correct"] = exp["is_correct"].apply(parse_correctness)

# ============================================================
# 4. VALIDATE AND ALIGN
# ============================================================
merged = base.merge(
    exp,
    on="_id",
    how="outer",
    suffixes=("_base", "_exp"),
    indicator=True,
    validate="one_to_one"
)

missing = merged[merged["_merge"] != "both"]

if not missing.empty:
    print("WARNING: Question IDs do not match.")
    print(missing[["_id", "_merge"]])
    raise ValueError("Resolve unmatched IDs before analysis.")

merged = merged.drop(columns="_merge")

# ============================================================
# 5. CHECK GOLD ANSWERS
# ============================================================
gold_base = merged["ground_truth_base"].astype(str).str.strip().str.lower()
gold_exp = merged["ground_truth_exp"].astype(str).str.strip().str.lower()

assert gold_base.equals(gold_exp), "Gold answers differ!"

# ============================================================
# 6. SEPARATE VALID ANSWERS AND FAILURES
# ============================================================
valid = merged[
    merged["is_correct_base"].notna()
    & merged["is_correct_exp"].notna()
].copy()

failed = merged[
    merged["is_correct_base"].isna()
    | merged["is_correct_exp"].isna()
].copy()

b = valid["is_correct_base"].astype(bool)
v = valid["is_correct_exp"].astype(bool)

# ============================================================
# 7. PAIRED TRANSITION ANALYSIS
# ============================================================
transitions = {
    "Wrong -> Correct": int((~b & v).sum()),
    "Correct -> Wrong": int((b & ~v).sum()),
    "Correct -> Correct": int((b & v).sum()),
    "Wrong -> Wrong": int((~b & ~v).sum())
}

print("\n=== PAIRED TRANSITION ANALYSIS ===")

for label, count in transitions.items():
    print(f"{label}: {count}")

n = len(valid)

if n:
    baseline_accuracy = b.mean()
    experimental_accuracy = v.mean()

    improved = transitions["Wrong -> Correct"]
    degraded = transitions["Correct -> Wrong"]

    print(f"\nValid paired questions: {n}")
    print(f"Failed/incomplete pairs: {len(failed)}")

    print(f"Baseline accuracy: {baseline_accuracy:.4f}")
    print(f"Experimental accuracy: {experimental_accuracy:.4f}")

    print(
        f"Absolute improvement: "
        f"{experimental_accuracy - baseline_accuracy:+.4f}"
    )

    print(f"Recovery count: {improved}")
    print(f"Over-correction count: {degraded}")

    print(
        f"Recovery rate: "
        f"{improved / (~b).sum():.4f}"
        if (~b).sum() else "Recovery rate: N/A"
    )

    print(
        f"Over-correction rate: "
        f"{degraded / b.sum():.4f}"
        if b.sum() else "Over-correction rate: N/A"
    )

    print(
        f"Net change among discordant cases: "
        f"{(improved - degraded) / n:+.4f}"
    )

# ============================================================
# 8. SAVE ANALYSIS
# ============================================================
valid.to_csv("paired_analysis_valid.csv", index=False)
failed.to_csv("paired_analysis_failed.csv", index=False)

print("\nSaved:")
print("- paired_analysis_valid.csv")
print("- paired_analysis_failed.csv")



=== PAIRED TRANSITION ANALYSIS ===
Wrong -> Correct: 0
Correct -> Wrong: 3
Correct -> Correct: 6
Wrong -> Wrong: 1

Valid paired questions: 10
Failed/incomplete pairs: 0
Baseline accuracy: 0.9000
Experimental accuracy: 0.6000
Absolute improvement: -0.3000
Recovery count: 0
Over-correction count: 3
Recovery rate: 0.0000
Over-correction rate: 0.3333
Net change among discordant cases: -0.3000

Saved:
- paired_analysis_valid.csv
- paired_analysis_failed.csv


In [24]:

from api_client import GeminiAPIClient
from evaluator import (
    load_hotpot,
    format_context,
    extract_answer,
    evaluate_prediction,
    evaluate_pair
)

# Initialize Gemini
client = GeminiAPIClient()

# Load dataset
dataset = load_hotpot(
    "/content/hotpot_dev_distractor_v1.json",
    limit=10
)

# Evaluate existing baseline and experimental CSVs
valid, failed, summary = evaluate_pair(
    "/content/baseline_results.csv",
    "/content/experimental_results.csv"
)



=== PAIRED EVALUATION ===
Total paired: 10
Valid pairs: 10
Incomplete pairs: 0
Baseline accuracy: 0.9
Experimental accuracy: 0.6
Absolute improvement: -0.30000000000000004
Wrong -> Correct: 0
Correct -> Wrong: 3
Correct -> Correct: 6
Wrong -> Wrong: 1
Recovery rate: 0.0
Over-correction rate: 0.3333333333333333
Saved: paired_analysis_valid.csv, paired_analysis_failed.csv
